In [1]:
import pandas as pd
import re
import struct
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

from scipy.signal import find_peaks
import matplotlib.colors as mcolors

current_dir = '/home/marian/CIGAR_ANALYSIS/CIGAR/notebooks'

# Build the absolute path to ../functions
functions_path = os.path.abspath(os.path.join(current_dir, '../functions'))

# Add it to sys.path
sys.path.append(functions_path)

import parse_data 
import cigar as cig

from tqdm import tqdm


/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
colors = ['royalblue', 'crimson', 'black', 'green', 'darkorange', 'brown', 'coral', 'indigo', 'magenta', 'blue']


##  Data reading

In [ ]:
gas = 'Xe'
temperature = 'room temp'
pressure = 6.5 # [bar]

In [ ]:
runs = []
for ii in range(92, 102):
    runs.append(f'Run{ii}')

print(runs)

['Run92', 'Run93', 'Run94', 'Run95', 'Run96', 'Run97', 'Run98', 'Run99', 'Run100', 'Run101']


In [ ]:

new_board = True

base        = f"/home/marian/CIGAR_ANALYSIS/CIGAR/data"

nchannels       = 10
nevents_per_wvf = 500
# nchannels       = 5
# nevents_per_wvf = 2000

# samples_per_waveform = 752
samples_per_waveform = 1124 if gas == 'Xe' else 2000
# samples_per_waveform = 2000
# samples_per_waveform = 500

event_header_bytes = 28
# event_header_bytes = 3036

# sample_binning = 1
sample_binning = 8e-9

start = 0
nfiles = 7

offline_trg = None
# offline_trg = 'AND'
# offline_trg = 'Majority3'

In [ ]:
if new_board:
    # amp_factors = {'CH1':(-1/360, 0),
    #                 'CH2':(-1/358, 0),
    #                 'CH3':(-1/359, 0),
    #                 'CH4':(-1/349, 0)
    #                 }
    amp_factors = {'CH1':(-1/350, 0),
                        'CH2':(-1/349, 0),
                        'CH3':(-1/350, 0),
                        'CH4':(-1/341, 0)
                        }

else:
    amp_factors = {'CH1':(1/269, 0),
                    'CH2':(1/267, 0),
                    'CH3':(1/258, 0),
                    'CH4':(1/275, 0)
                    }

In [ ]:

params = {'is_amplified':False
          ,'amp_factors':amp_factors
          ,'pes':True
          ,'temperature': temperature
          ,'pressure':pressure
          }

In [ ]:
polarity = 1
if new_board:
    if not params['is_amplified']:
        polarity = -1

In [ ]:
parse_data.checkWfs(run_dir, 
                    1, 10, 
                    nchannels, 
                    samples_per_waveform, 
                    event_header_bytes, 
                    print_headers = False
                    )

/home/marian/CIGAR_ANALYSIS/CIGAR/data/Run185
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)
Waveforms shape: (500, 10, 2000)


In [ ]:

# Initialize an empty list to store DataFrames
dataframes = []

# Loop through each folder and read all CSV files
for i, file in enumerate(tqdm(os.listdir(run_dir)[start:nfiles],desc="Reading .bin files", unit="file")):
    if file.endswith(".bin"):
        file_path = os.path.join(run_dir, file)
        # print(file_path)
        df = parse_data.parse_wf_from_binary(file_path, channels = nchannels, 
                                             n_events = nevents_per_wvf,
                                             file_idx = i,
                                             event_header_bytes = event_header_bytes
                                             )  
        dataframes.append(df)

Reading .bin files: 100%|██████████| 7/7 [00:37<00:00,  5.34s/file]


In [ ]:
# Merge all DataFrames into one
merged_df = pd.concat(dataframes, ignore_index=True)
merged_df.tail()

,TIME,CH1,CH2,CH3,CH4,CH5,CH6,CH7,CH8,CH9,CH10,event,event_time,file_idx
6999995,0.000016,2.406066,3.558068,-357.052307,413.451508,-803.727722,356.723114,-753.574097,373.817291,-448.958374,414.333282,3499,1636155592,6
6999996,0.000016,2.591148,3.558068,-320.710815,413.206512,-812.258423,356.172150,-758.726318,373.878662,-495.158386,414.947357,3499,1636155592,6
6999997,0.000016,2.714536,3.558068,-295.932495,412.900299,-822.630310,355.866058,-764.123901,374.185486,-536.621521,415.193024,3499,1636155592,6
6999998,0.000016,2.899618,3.680760,-281.554962,413.022797,-827.969666,355.866058,-765.902649,373.755951,-565.227356,415.008789,3499,1636155592,6
6999999,0.000016,2.467760,3.619414,-277.639374,413.390259,-830.056274,355.437500,-761.363770,373.326416,-582.144836,415.131592,3499,1636155592,6


In [ ]:
for run in runs:
    run_dir     = f"{base}/{run}"
